# 11장 · OpenAI Responses API의 요청·응답 구조

SDK는 Python에서 요청을 만들고 응답을 읽게 해 주는 라이브러리입니다. 고정 응답으로도 메시지, 출력 본문, 사용량 필드를 따로 살펴볼 수 있습니다.

Responses API 요청에서 질문의 역할과 내용을 지정하고, 수업용 응답에서 답변·완료 상태·사용량을 각각 읽습니다. 응답 일부만 받았을 때는 완료 상태가 해석 기준이 됩니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 httpx 0.28.1, openai 3.35.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 사용자 메시지의 role과 content

SDK는 파이썬 코드에서 서비스의 요청과 응답을 다루게 해 주는 라이브러리입니다. 먼저 보내려는 내용만 조립하면 실제 통신 없이도 모델 이름과 사용자 메시지가 어느 칸에 들어 있는지 볼 수 있습니다.


### API·SDK·토큰의 관계

<p>API는 프로그램이 서비스와 요청·응답을 주고받는 규약입니다. Python SDK는 그 규약을 Python 객체와 메서드로 사용하도록 돕는 라이브러리입니다. SDK를 설치하는 것과 서버에 모델 요청을 보내는 것은 별개의 일입니다.</p>

<p>모델은 요청의 입력을 처리하고 출력을 생성합니다. 토큰은 모델이 텍스트 등을 처리할 때 사용하는 단위이며 한국어 글자 수와 일치하지 않습니다. 사용량 필드의 입력 토큰·출력 토큰과 답변 글자 수를 같은 값으로 취급하지 않습니다.</p>

<p>컨텍스트는 모델이 이번 요청에서 참고할 입력 범위입니다. 긴 문서를 넣는다고 전체 내용을 정확히 기억한다는 뜻은 아닙니다. 모델 선택에는 기능 지원, 입력 길이, 응답 품질, 지연, 비용 조건을 함께 확인해야 합니다. 이번 교재는 특정 모델을 모든 과제에 최선이라고 정하지 않습니다.</p>

<table><caption>이번 예제의 책임</caption><thead><tr><th scope="col">요소</th><th scope="col">하는 일</th></tr>
</thead><tbody><tr><td>내 Python 코드</td><td>입력 구성·호출·출력 기록</td></tr>
<tr><td>OpenAI SDK</td><td>요청 직렬화·응답 객체 처리</td></tr>
<tr><td>실제 API 서버</td><td>선택 모델 실행</td></tr>
<tr><td>MockTransport</td><td>HTTP 전송 대신 고정 응답 반환</td></tr>
</tbody></table>


### 호출 전 입력 검증

<p>다음 장에서는 응답 형식을 스키마로 정하고, 불완전 응답·거절·스트리밍을 구분합니다. 오늘의 호출 함수와 오류 기록이 출발점입니다.</p>


### 셀 01 · role과 content 구성

model은 예제의 대역 이름입니다. 실제 계정에서 선택할 모델과 구분하세요.


In [1]:
question = "SDK가 하는 일을 한 문장으로 설명하세요."
request = {"model": "classroom-mock", "input": [{"role": "user", "content": question}], "max_output_tokens": 300}
print(request["input"][0]["content"])

SDK가 하는 일을 한 문장으로 설명하세요.


원래 질문이 그대로 나옵니다. max_output_tokens는 생성할 출력의 한도이고 입력 문장 길이나 실제 생성 토큰 수가 아닙니다.

이 셀을 여러 번 실행해도 호출 비용이 쌓이지 않습니다. 만든 것은 요청 내용을 담은 파이썬 사전이고 마지막 줄은 그 안의 질문을 읽습니다. 다음 단계에서 응답을 볼 때는 어떤 전송 함수가 그 값을 돌려줬는지도 확인하세요.


### 대화 메시지 목록 구성

<p>이 예제의 input에는 사용자 메시지 하나가 있습니다. 프로그램을 다시 실행해도 지난 요청의 내용이 자동으로 포함된다고 가정하지 않습니다. 후속 질문에 앞선 내용이 필요하면 그 내용을 입력으로 전달하는 방식을 명시적으로 구현해야 합니다.</p>


## MockTransport를 이용한 SDK 호출

아래는 제공 셀입니다. OpenAI SDK와 HTTPX가 설치된 커널에서는 SDK의 responses.create를 실제로 호출합니다. 전송은 로컬의 MockTransport가 받아 고정 응답을 돌려주므로 계정과 모델 서버를 쓰지 않습니다.


### 셀 02 · 모의 SDK 응답 생성

제공 함수는 요청에서 user 역할을 확인하고, message 하나와 사용량 예시가 든 HTTP 응답을 준비합니다.

제공 함수는 네트워크 대신 예제 응답을 만듭니다. 셀 03에서는 본문을, 과제 04에서는 같은 응답의 사용량을 읽습니다. 처음에는 함수 안의 생성 코드보다 반환된 응답 한 건을 따라가세요.


In [2]:
import json
from importlib.util import find_spec
sdk_available = find_spec("httpx") is not None and find_spec("openai") is not None
if sdk_available:
    import httpx
    from openai import OpenAI

def mock_response(request):
    body = json.loads(request.content)
    assert body["input"][0]["role"] == "user"
    return httpx.Response(200, json={
        "id": "resp_local", "object": "response", "created_at": 0,
        "model": "classroom-mock", "status": "completed",
        "output": [{"id": "msg_local", "type": "message",
                    "role": "assistant", "status": "completed",
                    "content": [{"type": "output_text", "text": "가상 안내를 확인했습니다.", "annotations": []}]}],
        "usage": {"input_tokens": 10, "output_tokens": 8, "total_tokens": 18},
    })

이 셀은 함수를 정의합니다. 아직 요청을 보내지 않았으므로 답변 출력이 없습니다.


### 모의 응답 환경 구성

<p>이 모의 실행은 SDK가 요청을 구성하고 응답을 읽는지 확인합니다. 실제 모델의 답변 품질·사용 가능 모델·계정 권한·요금·네트워크는 확인하지 않습니다.</p>


### 셀 03 · output_text 읽기

고정 응답에 적힌 안내 문장을 SDK가 꺼낸 값과 대조하세요.


In [3]:
if sdk_available:
    transport = httpx.MockTransport(mock_response)
    with OpenAI(api_key="classroom-placeholder", base_url="https://example.invalid/v1",
                http_client=httpx.Client(transport=transport), max_retries=0) as client:
        response = client.responses.create(**request)
    print(response.output_text)
else:
    print("OpenAI SDK와 HTTPX가 설치된 수업 커널에서 모의 호출을 실행하세요.")

가상 안내를 확인했습니다.


SDK가 꺼낸 본문은 ‘가상 안내를 확인했습니다.’입니다. 이 문장은 제공 함수의 응답 본문에서 왔습니다. 실제 서비스 모델의 생성 결과나 품질을 보여 주지는 않습니다.


### SDK 요청 함수

<p>ask는 클라이언트, 모델 이름, 질문을 입력받습니다. instructions에는 응답 방식, input에는 사용자 메시지를 전달합니다. max_output_tokens는 출력의 상한이며 항상 그만큼 출력하거나 같은 길이의 문장을 만든다는 뜻은 아닙니다.</p>

<p>함수 안에서 요청이 끝난 뒤 응답 상태와 텍스트를 반환합니다. 호출 한 번의 성공이 내용의 정확성까지 보장하지 않으므로 10장의 원문 대조 기준을 함께 적용합니다.</p>


## usage와 응답 완료 상태

토큰은 모델이 입력과 출력을 세는 단위입니다. 과금이나 지연을 추측하지 않고 응답에 실제 제공된 값을 읽습니다. 요청 상태가 미완료면 본문 길이만 보고 성공으로 기록하지 않습니다.


### 과제 04 · usage의 토큰 합계

제공 응답과 같은 사용량 사전입니다. 사용량이 보고된 경우 총 토큰 수를 출력하세요.


In [4]:
usage = {"input_tokens": 10, "output_tokens": 8, "total_tokens": 18}
# 응답의 총량을 꺼내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>usage = {&quot;input_tokens&quot;: 10, &quot;output_tokens&quot;: 8, &quot;total_tokens&quot;: 18}
print(usage[&quot;total_tokens&quot;])</code></pre>
<p>실행 결과</p><pre><code>18</code></pre>
<p>18은 제공 응답에 넣어 둔 수치입니다. 실제 요청의 경우에는 모델·입력마다 달라지며 사용량이 빠진 경우도 따로 남깁니다.</p>
</details>


### 응답 객체의 본문과 사용량

<p>Responses API의 output에는 텍스트 외의 항목도 들어갈 수 있습니다. 첫 번째 항목의 첫 번째 필드가 항상 답변이라고 가정하지 않고 SDK의 output_text를 사용합니다. status와 usage도 따로 확인합니다.</p>

<p>이번 모의 응답은 항상 “가상 안내를 확인했습니다.”를 반환합니다. 질문을 바꾸어도 같은 답이 나오는 것은 모델이 실행되지 않기 때문입니다. 이를 답변 모델의 성능으로 평가하면 안 됩니다.</p>

<table><caption>결과 필드</caption><thead><tr><th scope="col">필드</th><th scope="col">해석</th></tr>
</thead><tbody><tr><td>mode</td><td>mock 또는 live 실행 구분</td></tr>
<tr><td>status</td><td>응답의 완료 상태</td></tr>
<tr><td>answer</td><td>SDK가 모은 텍스트 출력</td></tr>
<tr><td>usage</td><td>서버 응답의 토큰 사용량 또는 없음</td></tr>
<tr><td>seconds</td><td>로컬에서 측정한 요청 대기 포함 시간</td></tr>
</tbody></table>


### 셀 05 · 미완료 상태 판별

통신은 끝났지만 서비스가 완료하지 못했다고 가정합니다. 본문을 화면에 내보내도 될까요?


In [5]:
partial = {"status": "incomplete", "text": "가상 안"}
if partial["status"] == "completed":
    print(partial["text"])
else:
    print("미완료 응답")

미완료 응답


‘미완료 응답’이 남습니다. 실제 호출 기록에는 모델, 상태, 사용량과 경과 시간을 구분해 두고 오류가 나면 키나 요청 원문 전체 대신 필요한 오류 종류를 기록합니다.


### 완료 상태를 확인한 뒤 본문 읽기

<p>예제의 partial에는 짧은 text가 있지만 status는 incomplete입니다. 미리 받은 글자가 있어도 완성된 답변으로 표시하기 전에는 상태를 확인해야 합니다. 이 셀의 출력은 ‘미완료 응답’입니다.</p>


## 돌아보기

질문 하나의 입력, 응답 본문, 사용량을 다시 짚어 보세요. 같은 글자를 요청해도 라이브 응답의 값은 달라질 수 있습니다. 다음 장에서는 완료 응답을 받았을 때도 그 안의 공지 형식과 내용을 다시 확인합니다.
